# 🧠 04-LLM · 26 VLM 进阶：Q-Former 压缩、AnyRes 与幻觉  > 关键词：Q-Former、图像 token 压缩、AnyRes、2D 位置编码、对比解码  > **纯代码进阶**：继续不部署模型，用 numpy 把"怎么把大图塞进上下文""怎么把 576 个视觉 token 压到 32 个""怎么抑制幻觉"全部算清楚。

## 本节要解决的问题  | # | 问题 | 对应内容 | |---|---|---| | 1 | 视觉 token 太多怎么压缩？ | §1 Q-Former + 手算 ✍️ | | 2 | 高分图怎么分块？token 数怎么算？ | §2 AnyRes + 手算 ✍️ | | 3 | 图像位置信息怎么给？（2D RoPE） | §3 位置编码 + 手算 ✍️ | | 4 | VLM 为什么会幻觉？怎么缓解？ | §4 对比解码 | | 5 | 训练三阶段与评估指标 | §5 深入原理 |

## 故事引入  336×336 的图切成 14×14 patch，一进来就是 **576 个 token**——比一段话还贵。LLaVA 硬拼，BLIP-2 用 **Q-Former** 把 576 压成 **32**；图片分辨率不够时用 **AnyRes** 动态分块。这些是真实 VLM 上线的核心工程，本章全部用 numpy 复现。

## 核心概念：图像 token 的三种命运  | 方案 | 代表 | 视觉 token 数(336×336) | 思想 | |---|---|---|---| | 直拼 | LLaVA-1.5 | 576 | 全部 patch 塞进 LLM | | 可学习查询压缩 | BLIP-2 (Q-Former) | 32 | 32 个 query 从 576 个 key 中"问出"要点 | | 分块拼 | LLaVA-1.5 AnyRes | 1728 (2 块) | 高分图拆块拼入，保细节 |   **代价权衡**：直拼最简但上下文贵；Q-Former 省 18 倍但丢细节（小字/位置）；AnyRes 保细节但费 token。

## ✍️ 手算 Q-Former 压缩：576 → 32 tokens  **设定**：336×336 图 → 576 个 patch token（key/value），Q-Former 放 32 个**可学习 query token**。  **第 1 步：压缩比**  $$\text{压缩比}=\frac{576}{32}=\mathbf{18}\ \text{倍}$$  **第 2 步：cross-attention 形状**  $$Q(32,d)\times K^\top(576,d)\to A(32,576)\xrightarrow{\text{加权求和}} \text{out}(32,d)$$  **第 3 步：那 32 个 query 从哪来？**  与文本无关——它们是**可学习参数**（不是图像的也不是文本的），训练让它们学会"问出图像里对任务有用的信息"，再拼到文本前喂 LLM。  **手算收获**：  - 压缩的本质：**信息瓶颈**——32 个向量被迫"概括"576 个向量，省上下文但必然丢细节（所以 Q-Former 系模型看图看"大意"）  - 576 是 336/14 的平方：$24\times24$ patch；分辨率 ↑ 二次方地涨 token——这就是 AnyRes 需要"分块"的原因  - Q-Former 常配 Flamingo gated cross-attn 思路：query 与 image key 交叉注意、与文本自注意  - 面试速答：BLIP-2 = 冻结视觉塔 + 冻结 LLM + 只训 Q-Former（最省算力的 VLM 训练范式之一）

In [ ]:
# ---- §1.1 验证：numpy 复算 Q-Former 压缩 ----
import numpy as np
patch_tokens, query_tokens = 576, 32
print(f'压缩比 = {patch_tokens}/{query_tokens} = {patch_tokens/query_tokens:.0f}x')
assert patch_tokens / query_tokens == 18

# cross-attn 小规模数值演示
d = 8
rng = np.random.default_rng(0)
q = rng.standard_normal((query_tokens, d))      # 32 个可学习 query
k = rng.standard_normal((patch_tokens, d))      # 576 个图像 key
scores = q @ k.T / np.sqrt(d)
A = np.exp(scores - scores.max(axis=1, keepdims=True))
A = A / A.sum(axis=1, keepdims=True)
out = A @ k
print(f'cross-attn: Q{q.shape} x K{k.shape} -> A{A.shape} -> out{out.shape}')
assert A.shape == (32, 576) and np.allclose(A.sum(axis=1), 1.0)
print('\n✓ 与手算一致')


## ✍️ 手算 AnyRes 动态分块：672×336 一张图多少 token？  **设定**：基准分辨率 336×336 = 576 tokens；672×336 的宽图。  **第 1 步：分块**  $$672\div336=2,\quad 336\div336=1\Rightarrow\text{切成 }2\ \text{个 }336\times336\ \text{分块}$$  **第 2 步：token 数**（LLaVA-1.5：分块 + 整图缩略各一份）  $$\text{tokens}=576\times(1+N_{\text{blocks}})=576\times(1+2)=\mathbf{1728}$$  **手算收获**：  - 分块同时保留"整图缩略图"（第 1 个 576）——模型既要看到全貌又要看清局部  - token 线性增长：2 块 1728、4 块 2880——**高分图 token 爆炸**，这是视觉 token 压缩存在的原因  - 单图 336×336 是 (1+1)×576=1152：LLaVA 基准其实也放了两份（缩略+原图/特征）  - 面试速答：AnyRes = 动态网格分块 + 缩略图；代价是 token 翻倍，收益是 OCR/细粒度理解大幅提升

In [ ]:
# ---- §2.1 验证：numpy 复算 AnyRes token ----
import numpy as np
base = 576   # 336x336 的 patch token 数
def anyres_tokens(h, w):
    nb = (h // 336) * (w // 336)
    return base * (1 + nb)
t = anyres_tokens(672, 336)
print(f'672x336 → 分块 {t//base-1} 个 → tokens = {t}')
assert t == 1728
t2 = anyres_tokens(336, 336)
print(f'336x336 → tokens = {t2}')
assert t2 == 1152
print('\n✓ 与手算一致')


## ✍️ 手算 2D 位置编码：patch(2,5) 的行/列旋转角  **设定**：24×24 patch 网格，RoPE 频率 $\theta_i=\text{pos}/10000^{2i/d}$，$d=8$。  **第 1 步：行位置（row=2）**  $$\theta_r=[\,2/10000^{0},\ 2/10000^{2/8},\ 2/10000^{4/8},\ 2/10000^{6/8}\,]=[\mathbf{2.0},\ \mathbf{0.2},\ 0.063,\ 0.02]$$  **第 2 步：列位置（col=5）**  $$\theta_c=[\mathbf{5.0},\ \mathbf{0.5},\ 0.158,\ 0.05]$$  **第 3 步：每个位置的旋转 = 行角 + 列角组合**——相邻 patch 位置编码不同，模型才能知道"哪个 patch 在它上面"。  **手算收获**：  - 文本 1D 位置只编码"第几个 token"；图像是 2D，拆成 **row 与 col 两个旋转角**（二维 RoPE）  - 低频维度（$\theta$ 小）编码大尺度空间、高频维度编码局部——与人眼"先看布局再看细节"同构  - 没有位置编码的图像 token = "一袋无序 patch"（洗牌不影响结果）——attention 是置换等变的  - 面试速答：ViT 用可学习位置嵌入或 2D-RoPE；LLaVA 直拼时位置继承 patch 的 2D 网格

In [ ]:
# ---- §3.1 验证：numpy 复算 2D 位置角度 ----
import numpy as np
d_pos = 8
def angles(pos):
    return np.array([pos / 10000**(2*i/d_pos) for i in range(d_pos//2)])
tr, tc = angles(2), angles(5)
print('行角(2):', np.round(tr, 4).tolist())
print('列角(5):', np.round(tc, 4).tolist())
assert abs(tr[0]-2.0) < 1e-6 and abs(tr[1]-0.2) < 1e-6
assert abs(tc[0]-5.0) < 1e-6 and abs(tc[1]-0.5) < 1e-6
assert not np.array_equal(tr, tc)   # 行列区分开
print('\n✓ 与手算一致')


## 深入原理：训练三阶段与幻觉  **BLIP-2 训练三阶段**：  1. 图文对齐（ITC：对比损失）+ 生成（ITM/ITG）——Q-Former 学到"如何问图像"  2. 冻结视觉塔+Q-Former，接冻结 LLM，训 Q-Former↔LLM 的桥接（Flamingo 式前置 token）  3. 指令微调（SFT）解锁 LLM 对齐人类问答  **LLaVA 两阶段**：① 冻结 ViT+LLM 只训投影层；② 图文指令 SFT 全量微调。  **幻觉（hallucination）**：模型描述图中不存在的东西——来源：训练数据配图文不对齐、自回归"惯性"、奖励函数偏"流畅"而非"忠实"。  **缓解手段**：  - **对比解码**：用"只看文本的弱模型"校准，保留图像信息增益  - **图像打分/规则校验**（存在性检查）  - **数据层面**：严格图文对齐清洗、负样本注入  - **评估**：CHAIR、POPE、MMHal-Bench

## 代码实战：对比解码如何修正选择  给出两句候选（A 含幻觉词但语言流利、B 忠实于图但略"生硬"），朴素解码选 A；加上"图像证据奖励"后，修正选 B——一行 logit 调整就是对比解码的灵魂。

In [ ]:
# ---- 对比解码：图像证据修正生成 ----
import numpy as np
# 两个候选句的平均 log 概率（词表层面简化）
cand_A = np.array([-0.5, -0.8, -0.6])      # 流利但含幻觉词（"穿红色外套"）
cand_B = np.array([-1.2, -1.0, -1.1])      # 忠实但平淡（"穿黑色外套"）
img_bonus = np.array([0.9, 0.8, 0.7])      # 视觉分支对 B 的证据优势
plain_a, plain_b = cand_A.mean(), cand_B.mean()
scored_a = plain_a
scored_b = plain_b + img_bonus.mean()
print(f'朴素解码: A={plain_a:.3f}, B={plain_b:.3f} → 选 A（幻觉）')
print(f'对比解码: A={scored_a:.3f}, B={scored_b:.3f} → 选 B（修正）')
assert plain_a > plain_b and scored_b > scored_a
print('✓ 图像证据把选票从幻觉句转移到忠实句')


## 面试考点  **Q1：Q-Former 为什么用 32 个 query？**  固定可学习 query 与图像 576 个 key 做交叉注意力，把"最有用的信息"浓缩到 32 个向量——上下文省 18 倍；32 是效果/成本折中的经验值。  **Q2：AnyRes 和 Q-Former 能一起用吗？**  能：先 AnyRes 分块涨细节（为看小字），再压缩；但压缩会丢刚补回的细节——实际按任务取舍。  **Q3：VLM 幻觉的根源？**  ① 训练数据图文错配；② 缺乏"不知道"机制（图像没有信息却硬答）；③ 解码偏向流畅。对比解码/负样本微调/解码约束各有缓解。  **Q4：怎么评估 VLM 的图文对齐质量？**  检索任务 Recall@K/MRR（§20 mm05 手算）；问答 MMBench/MMMU；幻觉 CHAIR/POPE；细粒度 OCR DocVQA。  **Q5：视觉 token 压缩的极限？**  1 token 也能（LLaMA-VID/Kosmos-2 部分方案），但信息瓶颈明显；可学习 query + 动态分配仍是主流折中。

## 小结与下一章预告  - 图像 token 三方案：直拼 576 / Q-Former 压到 32（18x）/ AnyRes 分块涨到 1728  - 手算闭环：压缩比 18、672×336→1728 tokens、patch(2,5) 的 2D 旋转角  - 幻觉可缓解：对比解码数值演示"选票转移"  - 至此 VLM 全链路（§25 骨架 + §26 进阶）闭环；可衔接 §20-多模态的 CLIP/ViT/BLIP 手算再深挖

## ✅ 自测清单  - [ ] 能手算 Q-Former 压缩比与 cross-attn 形状  - [ ] 能手算 AnyRes 分块数与 token 数  - [ ] 能手算 2D-RoPE 行列旋转角  - [ ] 能说出三种视觉 token 方案与代价  - [ ] 能讲清幻觉根源与对比解码原理